In [97]:
import pandas as pd

In [98]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")

In [99]:
df.sample(5)

,question,answer
4,What is the boiling point of water in Celsius?,100
37,Who was the first person to step on the Moon?,Armstrong
89,Which country is known for the Eiffel Tower?,France
60,Which country is home to the Great Wall?,China
26,Who discovered penicillin?,Alexander-Fleming


In [100]:
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")
    return text.split()

In [101]:
vocab = {"<UNK>":0}

In [102]:
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens = tokenized_question + tokenized_answer
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

In [103]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [104]:
len(vocab)

324

In [105]:
def text_to_indices(text, vocab):
    
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    
    return indexed_text

In [106]:
text_to_indices("What is campusx ? ", vocab)

[1, 2, 0]

In [107]:
from torch.utils.data import Dataset, DataLoader

In [108]:
import torch

class QADataset(Dataset):
    
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
    
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)
        

In [109]:
dataset = QADataset(df, vocab)

In [110]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [111]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [112]:
for question, answer in dataloader:
    print(question, answer)

tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([[259]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[  1,   2,   3, 146,  86,  19, 192, 193]]) tensor([[194]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor([[72]])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor([[131]])
tensor([[ 42, 101,   2,   3,  17]]) tensor([[102]])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([[124]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[260]])
tensor([[ 1,  2,  3, 17, 18, 19, 20, 21, 22]]) tensor([[23]])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([[173]])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([[114]])
tensor([[  1,   2,   3,   4,   5, 286]]) tensor([[287]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]])
tensor([[  1,

In [113]:
import torch.nn  as nn


In [114]:
class SimpleRNN(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)
        
    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))
        return output
        
        
        

In [115]:
epochs = 20
learning_rate = 0.001


In [116]:
model = SimpleRNN(len(vocab))

In [117]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [118]:
for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 524.754321
Epoch: 2, Loss: 456.130481
Epoch: 3, Loss: 376.394606
Epoch: 4, Loss: 315.029703
Epoch: 5, Loss: 262.855338
Epoch: 6, Loss: 215.045748
Epoch: 7, Loss: 171.965015
Epoch: 8, Loss: 133.496004
Epoch: 9, Loss: 102.147005
Epoch: 10, Loss: 77.569310
Epoch: 11, Loss: 59.159481
Epoch: 12, Loss: 46.080208
Epoch: 13, Loss: 36.664961
Epoch: 14, Loss: 29.623451
Epoch: 15, Loss: 24.534865
Epoch: 16, Loss: 20.795240
Epoch: 17, Loss: 17.612206
Epoch: 18, Loss: 15.104916
Epoch: 19, Loss: 13.040948
Epoch: 20, Loss: 11.325732


In [119]:
def predict(model, question, threshold=0.5):
    numerical_question = text_to_indices(question, vocab)
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)
    output = model(question_tensor)
    probs = torch.nn.functional.softmax(output, dim=1)
    value,index = torch.max(probs, dim=1)
    if value<threshold:
        print("I don't know")
    print(list(vocab.keys())[index])
    
    

In [121]:
predict(model, "Who discovered gravity?")

newton
